# 2. Build Global N-Gram DuckDB Store

This notebook builds the global exact chord n-gram vocabulary `V_n` from Chordonomicon, attaches each exact n-gram to its harmonic class `H_n`, and writes the durable result directly to DuckDB.

The default values use `n = 3, 4, 5, 6, 7, 8`, controlled by `NS` so the range can be narrowed or expanded later.

This is the expensive corpus-counting step. Notebook 3 validates and inspects the DuckDB database; notebooks 4 and 7 use that database for trend and document-term analysis.

## Pipeline Role

This notebook builds the global vocabularies and global occurrence counts, then stores them directly in DuckDB:

- `V_n`: exact chord `n`-gram vocabulary, with global counts `O_n`.
- `H_n`: harmonic/transposition-normalized vocabulary, with global counts `bar O_n`.

It does **not** build document-term matrices or stratified counts. Those depend on the stable IDs created here and are built later.

## Design

The notebook streams the raw Chordonomicon CSV in chunks and writes compact analysis tables directly to `data/processed/harmonic_trends.duckdb`:

- `exact_ngrams`: one row per exact chord n-gram in `V_n`.
- `harmonic_ngrams`: one row per harmonic class in `H_n`.
- `exact_to_harmonic`: the map from exact n-grams to harmonic classes.
- `song_ngram_summary`: one row per song with `n_chords` and `n_ngrams_*` counts only.
- `ngram_run_summary`: compact run totals by `n`.
- `harmonic_parse_errors`: diagnostics for n-grams that could not be encoded as `12 x n` matrices.

The tables include both `harmonic_id` and `harmonic_key`: `harmonic_id` is the compact lookup id, while `harmonic_key` is the explicit serialized matrix representation.

DuckDB is the source of truth after this notebook. CSV vocabulary files are no longer part of the main pipeline handoff.

In [1]:
from pathlib import Path
import importlib
import sys

import duckdb

CWD = Path.cwd()
ROOT = CWD.parent if (CWD / "utils").exists() else CWD
NOTEBOOK_DIR = ROOT / "notebooks"

# Force this repo's notebook utilities to win over any stale or external `utils` package.
sys.path = [p for p in sys.path if p != str(NOTEBOOK_DIR)]
sys.path.insert(0, str(NOTEBOOK_DIR))
for module_name in list(sys.modules):
    if module_name == "utils" or module_name.startswith("utils."):
        del sys.modules[module_name]

from utils import duckdb_store as ds
from utils import ngram_features as ng

ds = importlib.reload(ds)
ng = importlib.reload(ng)

expected_files = {
    "duckdb_store": (NOTEBOOK_DIR / "utils" / "duckdb_store.py").resolve(),
    "ngram_features": (NOTEBOOK_DIR / "utils" / "ngram_features.py").resolve(),
}
loaded_files = {
    "duckdb_store": Path(ds.__file__).resolve(),
    "ngram_features": Path(ng.__file__).resolve(),
}
assert loaded_files == expected_files, f"Imported wrong utility module(s): {loaded_files}; expected {expected_files}"
assert hasattr(ng, "first_root_normalized_key"), (
    f"Loaded {loaded_files['ngram_features']}, but it lacks first_root_normalized_key. "
    "Restart the kernel and rerun from the top."
)

{
    "python": sys.executable,
    "duckdb_version": duckdb.__version__,
    **{name: str(path) for name, path in loaded_files.items()},
}

{'python': '/usr/local/bin/python3',
 'duckdb_version': '1.5.2',
 'duckdb_store': '/Users/juansalinas/Documents/GitHub/harmonic-trends/notebooks/utils/duckdb_store.py',
 'ngram_features': '/Users/juansalinas/Documents/GitHub/harmonic-trends/notebooks/utils/ngram_features.py'}

In [2]:
RAW_PATH = ROOT / "data" / "raw" / "chordonomicon_v2.csv"
DB_PATH = ROOT / "data" / "processed" / "harmonic_trends.duckdb"

NS = tuple(range(3, 9))
CHORD_COL = "chords"
ID_COL = "id"
INCLUDE_BASS_IN_HARMONIC_KEY = True
CHUNKSIZE = 50_000
SHOW_PROGRESS = True
OVERWRITE = True

# Production default is the full corpus. For a smoke run, set MAX_ROWS = 10_000.
MAX_ROWS = None

assert RAW_PATH.exists(), RAW_PATH
ROOT, DB_PATH

(PosixPath('/Users/juansalinas/Documents/GitHub/harmonic-trends'),
 PosixPath('/Users/juansalinas/Documents/GitHub/harmonic-trends/data/processed/harmonic_trends.duckdb'))

## Harmonic Representative

For each exact n-gram, we build a binary `12 x n` pitch-class matrix: rows are pitch classes and columns are consecutive chords. We choose the representative in its `Z/12` modulation class by transposing the first chord root to C. Slash bass notes do not determine the representative, but when `INCLUDE_BASS_IN_HARMONIC_KEY=True` they are included in the 12-vector for their chord and shifted along with the other pitch classes.

In [3]:
example_a = ("C", "F", "G")
example_b = ("D", "G", "A")
example_slash = ("C/B", "F", "G")

key_a = ng.first_root_normalized_key(example_a, include_bass=INCLUDE_BASS_IN_HARMONIC_KEY)
key_b = ng.first_root_normalized_key(example_b, include_bass=INCLUDE_BASS_IN_HARMONIC_KEY)
key_slash = ng.first_root_normalized_key(example_slash, include_bass=INCLUDE_BASS_IN_HARMONIC_KEY)

key_a == key_b, key_a, key_b, key_slash

(True,
 '3:110000001000100010000101000010000001',
 '3:110000001000100010000101000010000001',
 '3:110000001000100010000101000010000101')

## Build Vocabularies and DuckDB Tables

Production runs use `MAX_ROWS = None`. For testing, temporarily set `MAX_ROWS = 10_000`, inspect `run_summary`, then restore `MAX_ROWS = None`.

This is the longest notebook 2 cell. During the raw-song scan, `tqdm` shows song/chunk progress and live unique vocabulary sizes. After scanning, the DuckDB writer prints stage updates while it creates tables and indexes. On the full corpus, the indexing stage can take several minutes.

In [4]:
results = ng.build_ngram_vocabularies_from_csv(
    RAW_PATH,
    chord_col=CHORD_COL,
    id_col=ID_COL,
    ns=NS,
    include_bass=INCLUDE_BASS_IN_HARMONIC_KEY,
    chunksize=CHUNKSIZE,
    max_rows=MAX_ROWS,
    show_progress=SHOW_PROGRESS,
)

exact_vocab = results["exact_vocab"]
harmonic_vocab = results["harmonic_vocab"]
song_summary = results["song_summary"]
parse_errors = results["parse_errors"]
run_summary = ng.ngram_run_summary(exact_vocab, harmonic_vocab, song_summary, parse_errors, ns=NS)

db_path = ds.create_harmonic_trends_database_from_dataframes(
    db_path=DB_PATH,
    exact_vocab=exact_vocab,
    song_summary=song_summary,
    run_summary=run_summary,
    parse_errors=parse_errors,
    overwrite=OVERWRITE,
    metadata={
        "raw_path": str(RAW_PATH),
        "ns": ",".join(str(n) for n in NS),
        "include_bass_in_harmonic_key": str(INCLUDE_BASS_IN_HARMONIC_KEY),
    },
)

{
    "database": str(db_path),
    "exact_vocab_rows": len(exact_vocab),
    "harmonic_vocab_rows": len(harmonic_vocab),
    "song_summary_rows": len(song_summary),
    "parse_error_rows": len(parse_errors),
}

Processing chord rows: 14chunk [13:25, 57.55s/chunk, rows=679807, exact=3.44e+7, harmonic=2.07e+7, errors=0]
Counting n-grams: 100%|██████████| 679807/679807 [13:25<00:00, 843.74song/s]


Creating DuckDB database: /Users/juansalinas/Documents/GitHub/harmonic-trends/data/processed/harmonic_trends.duckdb
Registering exact vocabulary rows: 34,464,863
Building harmonic_ngrams from exact_ngrams...
Normalizing harmonic frequencies...
Creating exact_to_harmonic map and frequency validation...
Writing song_ngram_summary rows: 679,807
Writing ngram_run_summary rows: 6
Writing harmonic_parse_errors rows: 0
Creating metadata and reserved distance table...
Creating indexes. This can take several minutes on the full corpus...
DuckDB global n-gram store complete.


{'database': '/Users/juansalinas/Documents/GitHub/harmonic-trends/data/processed/harmonic_trends.duckdb',
 'exact_vocab_rows': 34464863,
 'harmonic_vocab_rows': 20696047,
 'song_summary_rows': 679807,
 'parse_error_rows': 0}

## Inspect Results

Check this summary before trusting a full run. The main warning signs are large parse-error rates or very fast growth in `unique_exact_ngrams` for `n = 7, 8`.

In [5]:
run_summary

,n,total_windows,exact_occurrences,unique_exact_ngrams,harmonic_occurrences,unique_harmonic_classes,parse_error_occurrences,parse_error_rate
0,3,50635145,50635145,828417,50635145,192951,0,0.0
1,4,49955522,49955522,2250695,49955522,843307,0,0.0
2,5,49276039,49276039,4269570,49276039,2081609,0,0.0
3,6,48596846,48596846,6614486,48596846,3802821,0,0.0
4,7,47918358,47918358,9060200,47918358,5815220,0,0.0
5,8,47240720,47240720,11441495,47240720,7960139,0,0.0


In [6]:
exact_vocab.head(20)

,exact_ngram_id,n,ngram_json,ngram,count,frequency,harmonic_key,harmonic_id
0,3_ce87d78ae19f8322,3,"[""G"", ""C"", ""G""]",G C G,522905,0.010327,3:111000000000101010000101000010000000,H3_ede3c4f53675bbb0
1,3_6adafd20068442f4,3,"[""C"", ""G"", ""D""]",C G D,459111,0.009067,3:100000011000100000001110000001000010,H3_4e99eecf61b0edec
2,3_8236526a0d85f6fd,3,"[""C"", ""G"", ""C""]",C G C,451349,0.008914,3:101000010000101000000111000000000010,H3_35d6bfda85b78e3b
3,3_b2f9fe8f46eb2ef6,3,"[""F"", ""C"", ""G""]",F C G,403830,0.007975,3:100000011000100000001110000001000010,H3_4e99eecf61b0edec
4,3_fe4dbe8dc705a25b,3,"[""D"", ""G"", ""D""]",D G D,396222,0.007825,3:111000000000101010000101000010000000,H3_ede3c4f53675bbb0
5,3_6458674f98ac0103,3,"[""G"", ""D"", ""G""]",G D G,371850,0.007344,3:101000010000101000000111000000000010,H3_35d6bfda85b78e3b
6,3_777c093b9cc665f8,3,"[""C"", ""F"", ""C""]",C F C,327305,0.006464,3:111000000000101010000101000010000000,H3_ede3c4f53675bbb0
7,3_c7ddb17876ab152a,3,"[""C"", ""D"", ""G""]",C D G,308646,0.006095,3:100000011000100000010101000010000001,H3_5f9ccd30c6b35272
8,3_2f503f769e5bef54,3,"[""C"", ""G"", ""Amin""]",C G Amin,307330,0.006069,3:101000010000101000000110000001000010,H3_aef0281c29774fb7
9,3_bc83194452c9c2a7,3,"[""G"", ""D"", ""A""]",G D A,305479,0.006033,3:100000011000100000001110000001000010,H3_4e99eecf61b0edec


In [7]:
harmonic_vocab.head(20)

,harmonic_id,n,harmonic_key,count,frequency,example_ngram_json,example_ngram
0,H3_ede3c4f53675bbb0,3,3:111000000000101010000101000010000000,2203877,0.043525,"[""C"", ""F"", ""C""]",C F C
1,H3_35d6bfda85b78e3b,3,3:101000010000101000000111000000000010,1907266,0.037667,"[""F"", ""C"", ""F""]",F C F
2,H3_4e99eecf61b0edec,3,3:100000011000100000001110000001000010,1827464,0.036091,"[""F"", ""C"", ""G""]",F C G
3,H3_5f9ccd30c6b35272,3,3:100000011000100000010101000010000001,1273259,0.025146,"[""C"", ""D"", ""G""]",C D G
4,H3_614d8a0823c338e1,3,3:110000001000100011000100000010001000,1183031,0.023364,"[""G"", ""C"", ""F""]",G C F
5,H3_aef0281c29774fb7,3,3:101000010000101000000110000001000010,1138503,0.022484,"[""G"", ""D/Fs"", ""Emin""]",G D/Fs Emin
6,H3_0859351c7b2e5686,3,3:101000010000100011000100000001010000,1031824,0.020378,"[""E"", ""D"", ""A/Cs""]",E D A/Cs
7,H3_cb0f464e362cf3bc,3,3:110000000111000000000101010000001000,962685,0.019012,"[""Gsmin"", ""E"", ""B""]",Gsmin E B
8,H3_2bd39a67fe3535a8,3,3:101000010000100001000110000001000010,924372,0.018256,"[""A/Cs"", ""E"", ""D""]",A/Cs E D
9,H3_950fb5de7bd72a8d,3,3:100000011000100011000100000010001000,878822,0.017356,"[""B"", ""Csmin"", ""A""]",B Csmin A


In [8]:
song_summary.head()

,n_chords,id,n_ngrams_3,n_ngrams_4,n_ngrams_5,n_ngrams_6,n_ngrams_7,n_ngrams_8
0,67,1,65,64,63,62,61,60
1,122,2,120,119,118,117,116,115
2,56,3,54,53,52,51,50,49
3,138,4,136,135,134,133,132,131
4,39,5,37,36,35,34,33,32


In [9]:
parse_errors.head(20)

,n,error_type,message,count,song_id,ngram_json,ngram


## Validate DuckDB Output

DuckDB is the durable artifact from this notebook. The checks below validate mass preservation and confirm that the core tables were created.

In [10]:
con = duckdb.connect(str(DB_PATH), read_only=True)
ds.configure_connection(con)
try:
    table_counts = con.execute(
        """
        SELECT 'exact_ngrams' AS table_name, COUNT(*) AS row_count FROM exact_ngrams
        UNION ALL SELECT 'harmonic_ngrams', COUNT(*) FROM harmonic_ngrams
        UNION ALL SELECT 'exact_to_harmonic', COUNT(*) FROM exact_to_harmonic
        UNION ALL SELECT 'song_ngram_summary', COUNT(*) FROM song_ngram_summary
        UNION ALL SELECT 'ngram_run_summary', COUNT(*) FROM ngram_run_summary
        UNION ALL SELECT 'harmonic_parse_errors', COUNT(*) FROM harmonic_parse_errors
        UNION ALL SELECT 'frequency_validation', COUNT(*) FROM frequency_validation
        ORDER BY table_name
        """
    ).fetchdf()
    validation = con.execute("SELECT * FROM frequency_validation ORDER BY n").fetchdf()
finally:
    con.close()

table_counts, validation

(              table_name  row_count
 0           exact_ngrams   34464863
 1      exact_to_harmonic   34464863
 2   frequency_validation          6
 3        harmonic_ngrams   20696047
 4  harmonic_parse_errors          0
 5      ngram_run_summary          6
 6     song_ngram_summary     679807,
    n   V_count   H_count  V_frequency_sum  H_frequency_sum  counts_match
 0  3  50635145  50635145              1.0              1.0          True
 1  4  49955522  49955522              1.0              1.0          True
 2  5  49276039  49276039              1.0              1.0          True
 3  6  48596846  48596846              1.0              1.0          True
 4  7  47918358  47918358              1.0              1.0          True
 5  8  47240720  47240720              1.0              1.0          True)

## Handoff

The deliverable is `data/processed/harmonic_trends.duckdb`.

Notebook 3 validates and inspects this database. Notebook 4 uses it for target-limited trend tracking. Notebook 7 uses it to build broad harmonic document-term counts.